In [2]:
def prime_factorization(n):
    factors = {}
    p = 2
    while p * p <= n:
        while n % p == 0:
            factors[p] = factors.get(p, 0) + 1
            n = n // p
        p += 1
    if n > 1:
        factors[n] = factors.get(p, 0) +  1
    return factors 

In [3]:
def extended_gcd(a, b):
    # computes the extended gcd
    # finds integers x and y such that: a*x + b*y = gcd(a, b)
    # return gcd(a, b), x, y
    if b == 0:
        return a, 1, 0
    g, x_i, y_i = extended_gcd(b, a % b)
    x = y_i
    y = x_i - (a // b) * y_i
    return g, x, y

def modular_inverse(a, m):
    # computes the modular multiplicative inverse of a modulo m
    # finds the number a^(-1) such that: a * a^(-1) = 1 (mod m)
    # if gcd(a, m) = 1, then a*x + m*y = 1 -> a*x = 1 (mod m). 
    # Therefore, x is the modular inverse of a modulo m
    a %= m
    g, x, _ = extended_gcd(a, m)
    if g == 1:
        return x % m 


In [4]:
from math import isqrt

def bsgs_dlog(alpha, beta, n):
    # Baby-step giant-stept algorithm for computing discrete logarithms
    # the key idea is to split the unknown exponent x as: x = i * m + j
    # we substitude x = i * m + j from: alpha ^ x = beta mod p
    # we obtain: alpha ^ j = beta * alpha ^ (-i * m) mod p
    # Output: the discrete logarithm x = log_alpha_beta
    p = n + 1
    m = isqrt(n)
    if m * m < n:
        m += 1

    table = {}
    alpha_j = 1
    for j in range(m):
        if alpha_j not in table:
            table[alpha_j] = j
        alpha_j = (alpha_j * alpha) % p

    alpha_m = pow(alpha, m, p)
    alpha_minus_m = modular_inverse(alpha_m, p)
    gamma = beta % p

    for i in range(m):
        if gamma in table:
            j = table[gamma]
            x = i * m + j
            if x < n and pow(alpha, x, p) == beta % p:
                return x
        
        gamma = (gamma * alpha_minus_m) % p

In [5]:
def crt_pair(a1, m1, a2, m2):
    g, s, t = extended_gcd(m1, m2)
    x = (a1 * t * m2 + a2 * s * m1) % (m1 * m2)
    return x, m1 * m2

def gauss_crt(congruences):
    x, M = congruences[0]
    for a, m in congruences[1:]:
        x, M = crt_pair(x, M, a, m)
    return x, M

In [6]:
def pohlig_hellman_algorithm(alpha, beta, n):
    p = n + 1
    # find the prime factorization of n
    factors = prime_factorization(n)

    congruences = []
    for q, e in factors.items():
        q_pow_e = q ** e

        gamma = 1
        alpha_bar = pow(alpha, n // q, p)
        
        # x_i = l_0 + l_1 * q + l_2 * q^2 + ... + l_(e-1) * q^(e - 1)
        x_i = 0
        for j in range(e):
            gamma_inv = modular_inverse(gamma, p)

            beta_bar = pow((beta * gamma_inv) % p, n // (q ** (j + 1)), p)

            l_j = bsgs_dlog(alpha_bar, beta_bar, n)

            gamma = (gamma * pow(alpha, l_j * (q ** j), p)) % p

            x_i += l_j * (q ** j) 

        congruences.append((x_i % q_pow_e, q_pow_e))
            
    x, _ = gauss_crt(congruences)
    return x % n

In [16]:
alpha = 2
x_true = 123
p = 1019
beta = pow(alpha, x_true, p)
x = pohlig_hellman_algorithm(alpha, beta, p - 1)
print("x = ", x)

x =  123
